In [1]:
import torch
import torch.nn as nn
import numpy as np

# Preparing data

In [2]:
file = np.load('DataTrain.npy')
data = torch.tensor(file, dtype=torch.float32)
features = data[:,:-1] #getting just the features
fraud = data[:, -1] #just the fraud
dataset = torch.utils.data.TensorDataset(features, fraud) #creating a dataset with separate features and fraud
loader = torch.utils.data.DataLoader(dataset, batch_size=1024, shuffle=True) #separates the dataset into clusters so we can feed it just enough info

# Creatring a NN class and defining the structure 

In [3]:
class Model(nn.Module):
  def __init__(self, layerSize = [128, 64, 32, 16]):
    super(Model, self).__init__()
    layers = []
    c = features.shape[1]
    for s in layerSize:
      layers.append(nn.Linear(c, s))
      layers.append(nn.ReLU())
      c = s
    layers.append(nn.Linear(c, 1))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

# Creating an object of NN and defining loss function

In [4]:
model = Model()
pos_weight = (len(fraud) - fraud.sum()) / fraud.sum() #finding an appropriate weight cuz fraud is ver rare, and it will just generate 0 all the time
optimiser = torch.optim.Adam(model.parameters(), lr=0.008) #setting learning rate and the optimisation function
loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight) #setting loss function

# Training my boy 

In [ ]:
total_loss=0
num_batches = 0
for epoch in range(40):
    for feature,is_fraud in loader:
        optimiser.zero_grad() #reset gradient
        
        outputs = model(feature).squeeze() #get the predictions 
        is_fraud = is_fraud.float() #convert to float cuz it likes float 
        
        angry_loss_value = loss(outputs, is_fraud) #checks how actually accurate it is 
        angry_loss_value.backward() #holy fricking legend of a back propagation

        optimiser.step() #adjusts the weights so it actually learns something 
        #Everything after this point of code in this cell is just looking at the progress made by the model
        total_loss += angry_loss_value.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch}, Avg Loss: {avg_loss}")
    total_loss=0
    num_batches=0

Epoch 0, Avg Loss: 0.6818026591212898


# Testing how well my boy performs

In [ ]:
correct = 0
with torch.no_grad(): #tell the model not to use the gradient cuz we are no training anymore 
    sample1, sample2 = next(iter(loader)) #random samples of data
    outputs=model(sample1).squeeze() #get the prediction
    probis = torch.sigmoid(outputs) #putting the prediction through the sigmoid function to get an actual probability because just outputs is an array of nonsense
    predis = (probis>0.945).float() #very basic way to find who is a fraud
    for i in range(len(sample2)):
        if(sample2[i].float()==1.0)and(sample2[i]==predis[i]):
            correct+=1
        if(sample2[i].float()==1.0):
            print('Actual fraud:',sample2[i].float(),predis[i].float(),probis[i].float())
        if(predis[i].float()==1.0) and (sample2[i].float()!=1):
            print('Predicted fraud:',sample2[i].float(),predis[i].float(),probis[i].float())
    predis_sum = predis.sum().item()
    actual_sum = sample2.sum().item()
    print("\nPredicted fraud:", predis_sum)
    print("Actual fraud:", actual_sum)
    print('Correct:',correct)
    print('Accuracy:', correct/predis_sum*100)